Setup & Load Data

In [28]:
import pandas as pd

# Load data
df = pd.read_csv("../data/processed/steam_tableau.csv")

# Ensure timestamp is datetime
df["timestamp"] = pd.to_datetime(df["timestamp"])

# Filter to today's date
df_today = df[df["timestamp"].dt.date == pd.to_datetime("2026-04-16").date()]
df_today = df_today[df_today["item_type"].str.lower() == "game"]

# Filter for: Today's date, only Games, and only the Top 200 positions
df_top200 = df_today[df["rank_position"] <= 200]

# Separate datasets
topsellers = df_today[df_today["rank_type"] == "topsellers"].copy()
popularnew = df_today[df_today["rank_type"] == "popularnew"].copy()

print("Top Sellers:", topsellers.shape)
print("New Releases:", popularnew.shape)

Top Sellers: (1767, 17)
New Releases: (447, 17)


C:\Users\catwi\AppData\Local\Temp\ipykernel_91280\2252487643.py:14: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_top200 = df_today[df["rank_position"] <= 200]


Genre Distribution (Top Sellers vs New Releases)

Goal:

Which genres dominate each ranking?

In [29]:
def genre_distribution(df, name):

    genre_counts = (
        df["genre"]
        .value_counts()
        .reset_index()
    )
    genre_counts.columns = ["genre", "count"]

    print(f"\n{name} - Top Genres:")
    print(genre_counts.head(10))

    return genre_counts

ts_genres = genre_distribution(topsellers, "Top Sellers")
nr_genres = genre_distribution(popularnew, "Popular New")


Top Sellers - Top Genres:
                   genre  count
0                 Action    467
1              Adventure    338
2                    RPG    260
3             Simulation    254
4               Strategy    173
5                 Casual    104
6  Massively Multiplayer     97
7                 Sports     43
8                 Racing     21
9   Animation & Modeling      2

Popular New - Top Genres:
                   genre  count
0                 Action    101
1              Adventure     90
2             Simulation     83
3                    RPG     56
4               Strategy     51
5                 Casual     43
6                 Sports     11
7  Massively Multiplayer      9
8                 Racing      3


In [30]:
print("Columns in df_top200:", df_top200.columns.tolist())
print("Unique rank_types found:", df_top200["rank_type"].unique().tolist())

# ANALYSIS FUNCTION
def analyze_genre_by_list(df, target_rank_type):
    mask = df["rank_type"].str.lower() == target_rank_type.lower()
    subset = df[mask]

    if subset.empty:
        print(f"\n[!] Result for '{target_rank_type}' is EMPTY.")
        return None

    # Calculate counts
    genre_counts = subset["genre"].value_counts().reset_index()
    genre_counts.columns = ["Genre", "Count"]

    # Calculate unique games (using app_id)
    unique_games = subset["appid"].nunique()

    print(f"\n--- {target_rank_type.upper()} ---")
    print(f"Total Unique Games: {unique_games}")
    print(genre_counts.head(10))

    return genre_counts

# 4. EXECUTION
ts_results = analyze_genre_by_list(df_top200, "topsellers")
nr_results = analyze_genre_by_list(df_top200, "popularnew")

Columns in df_top200: ['appid', 'title', 'item_type', 'release_date', 'developer', 'publisher', 'price', 'user_positive_total', 'rank_type', 'rank_position', 'timestamp', 'indie', 'free_to_play', 'early_access', 'singleplayer', 'multiplayer', 'genre']
Unique rank_types found: ['topsellers', 'popularnew']

--- TOPSELLERS ---
Total Unique Games: 199
                   Genre  Count
0                 Action    241
1              Adventure    172
2                    RPG    131
3             Simulation    126
4               Strategy     89
5  Massively Multiplayer     56
6                 Casual     45
7                 Sports     22
8                 Racing     10

--- POPULARNEW ---
Total Unique Games: 110
                   Genre  Count
0                 Action    101
1              Adventure     90
2             Simulation     83
3                    RPG     56
4               Strategy     51
5                 Casual     43
6                 Sports     11
7  Massively Multiplayer      

Relationship: Rank vs Total Positive Ratings

Goal:

Do higher-ranked games have more positive ratings?

In [31]:
def rank_vs_ratings(df, name):
    subset = df[["rank_position", "user_positive_total"]].dropna()

    correlation = subset["rank_position"].corr(subset["user_positive_total"])

    print(f"\n{name} Correlation (Rank vs Total Positive Ratings): {correlation:.3f}")

    return subset

ts_corr_data = rank_vs_ratings(topsellers, "Top Sellers")
nr_corr_data = rank_vs_ratings(popularnew, "Popular New")

from scipy import stats

def rank_vs_ratings(df, name):
    # Drop NAs to ensure the arrays match in length
    subset = df[["rank_position", "user_positive_total"]].dropna()

    # Calculate Pearson correlation and p-value
    r_val, p_val = stats.pearsonr(subset["rank_position"], subset["user_positive_total"])

    print(f"\n--- {name} Correlation Analysis ---")
    print(f"Correlation (r): {r_val:.3f}")

    # Format p-value: scientific notation if very small, otherwise 3 decimals
    if p_val < 0.001:
        print(f"P-value: {p_val:.3e} (Statistically Significant)")
    else:
        print(f"P-value: {p_val:.3f}")

    return subset

ts_corr_data = rank_vs_ratings(topsellers, "Top Sellers")
nr_corr_data = rank_vs_ratings(popularnew, "Popular New")


Top Sellers Correlation (Rank vs Total Positive Ratings): 0.113

Popular New Correlation (Rank vs Total Positive Ratings): 0.082

--- Top Sellers Correlation Analysis ---
Correlation (r): 0.113
P-value: 2.337e-06 (Statistically Significant)

--- Popular New Correlation Analysis ---
Correlation (r): 0.082
P-value: 0.084


Genre + Ratings Combined

Goal:

Which genres have both:

-high presence
-high ratings

In [32]:
def genre_rating_analysis(df, name):
    result = (
        df.groupby("genre")
        .agg(
            avg_ratings=("user_positive_total", "mean"),
            count=("appid", "count")
        )
        .reset_index()
        .sort_values("count", ascending=False)
    )

    print(f"\n{name} - Genre Performance:")
    print(result.head(10))

    return result

ts_genre_perf = genre_rating_analysis(topsellers, "Top Sellers")
nr_genre_perf = genre_rating_analysis(popularnew, "Popular New")


Top Sellers - Genre Performance:
                    genre  avg_ratings  count
0                  Action    82.316018    467
1               Adventure    83.961788    338
7                     RPG    82.029730    260
9              Simulation    84.626880    254
11               Strategy    84.661628    173
3                  Casual    86.984020    104
5   Massively Multiplayer    73.894639     97
10                 Sports    73.495250     43
8                  Racing    86.336667     21
4   Design & Illustration    98.120000      2

Popular New - Genre Performance:
                   genre  avg_ratings  count
0                 Action    86.335842    101
1              Adventure    86.886889     90
6             Simulation    87.797952     83
4                    RPG    84.362500     56
8               Strategy    86.620196     51
2                 Casual    89.640233     43
7                 Sports    87.157273     11
3  Massively Multiplayer    76.670000      9
5                 Rac

Genre Overperformance Analysis

In [34]:
for rank_type, subset in df.groupby('rank_type'):
    print(f"\n--- {rank_type.upper()} ---")

    genre_avg_rank = subset.groupby('genre')['rank_position'].mean()
    overall_avg = subset['rank_position'].mean()

    genre_performance = overall_avg - genre_avg_rank

    print(genre_performance.sort_values(ascending=False))


--- POPULARNEW ---
genre
Strategy                  4.483310
Simulation                3.626354
Casual                    1.399359
Adventure                 1.287843
RPG                      -2.043904
Action                   -2.300443
Sports                  -13.765392
Racing                  -16.464738
Massively Multiplayer   -18.481766
Name: rank_position, dtype: float64

--- TOPSELLERS ---
genre
Massively Multiplayer    15.238896
Sports                    7.366576
Action                    5.871511
Adventure                 3.150440
RPG                      -3.966700
Simulation               -5.284007
Strategy                 -7.839548
Racing                   -9.327160
Casual                  -11.629330
Design & Illustration   -73.285494
Animation & Modeling    -73.285494
Photo Editing           -73.285494
Utilities               -73.285494
Name: rank_position, dtype: float64


Success Profile” Modeling (Nonlinear Rank Effects)

In [35]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

for rank_type, subset in df.groupby('rank_type'):

    print(f"\n==============================")
    print(f"MODEL FOR: {rank_type.upper()}")
    print(f"==============================")

    # -------------------------
    # Nonlinear target
    # -------------------------
    y = 1 / subset['rank_position']

    # -------------------------
    # Price conversion (temporary)
    # -------------------------
    price_numeric = (
        subset['price']
        .replace('Free', np.nan)
        .str.replace('$', '', regex=False)
        .pipe(pd.to_numeric, errors='coerce')
    )

    # -------------------------
    # Feature engineering
    # -------------------------
    is_free = (subset['price'] == 'Free').astype(int)

    X = pd.DataFrame({
        'user_positive_total': subset['user_positive_total'],
        'price_numeric': price_numeric,
        'is_free': is_free
    })

    X = sm.add_constant(X)

    # -------------------------
    # Fit model
    # -------------------------
    model = sm.OLS(y, X, missing='drop').fit()

    print(model.summary())


MODEL FOR: POPULARNEW
                            OLS Regression Results                            
Dep. Variable:          rank_position   R-squared:                       0.017
Model:                            OLS   Adj. R-squared:                  0.016
Method:                 Least Squares   F-statistic:                     14.66
Date:                Thu, 16 Apr 2026   Prob (F-statistic):           4.86e-07
Time:                        21:53:17   Log-Likelihood:                 1400.0
No. Observations:                1712   AIC:                            -2794.
Df Residuals:                    1709   BIC:                            -2778.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------
const      

C:\Users\catwi\PycharmProjects\PythonProject_steam-game-insights\.venv\Lib\site-packages\statsmodels\regression\linear_model.py:1966: RuntimeWarning: divide by zero encountered in scalar divide
  return np.sqrt(eigvals[0]/eigvals[-1])
C:\Users\catwi\PycharmProjects\PythonProject_steam-game-insights\.venv\Lib\site-packages\statsmodels\regression\linear_model.py:1966: RuntimeWarning: divide by zero encountered in scalar divide
  return np.sqrt(eigvals[0]/eigvals[-1])


Interaction Effects

In [36]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

for rank_type, subset in df.groupby('rank_type'):

    print(f"\n==============================")
    print(f"INTERACTION MODEL FOR: {rank_type.upper()}")
    print(f"==============================")

    df_temp = subset.copy()

    # -----------------------------
    # 1. Convert price -> numeric
    # -----------------------------
    df_temp['price_numeric'] = (
        df_temp['price']
        .replace('Free', np.nan)
        .str.replace('$', '', regex=False)
        .pipe(pd.to_numeric, errors='coerce')
    )

    # -----------------------------
    # 2. Nonlinear target
    # -----------------------------
    df_temp['success'] = 1 / df_temp['rank_position']

    # -----------------------------
    # 3. Interaction model
    # -----------------------------
    model = smf.ols(
        """
        success ~
            user_positive_total * C(genre) +
            price_numeric * C(genre) +
            free_to_play +
            indie +
            multiplayer +
            singleplayer +
            early_access
        """,
        data=df_temp
    ).fit()

    print(model.summary())


INTERACTION MODEL FOR: POPULARNEW
                            OLS Regression Results                            
Dep. Variable:                success   R-squared:                       0.049
Model:                            OLS   Adj. R-squared:                  0.032
Method:                 Least Squares   F-statistic:                     2.793
Date:                Thu, 16 Apr 2026   Prob (F-statistic):           6.04e-07
Time:                        21:55:08   Log-Likelihood:                 1430.9
No. Observations:                1707   AIC:                            -2798.
Df Residuals:                    1675   BIC:                            -2624.
Df Model:                          31                                         
Covariance Type:            nonrobust                                         
                                                            coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------